# [실습] 군집 — 고객을 묶고 의심하기

_이 노트북은 LMS에서 내보냈습니다. 영상·퀴즈는 학습 참고용으로 마크다운으로 변환되었습니다._

# Chapter 6 실습. 고객을 몇 개 그룹으로 나눠야 할까?

**필수 머신러닝 · Chapter 6**

| Part | 주제 |
| --- | --- |
| Part 1 | 고객을 숫자로: 정제와 RFM |
| Part 2 | K-means로 묶기 |
| Part 3 | 묶고 의심하기 |
| Part 4 | CRM팀에 보고하기 |

## 분석 의뢰서

> **의뢰 부서**: 온라인 도매몰 CRM팀
> **목적**: 고객 4,000여 명을 몇 개 그룹으로 나눠 그룹별로 다른 관리 정책(할인, 리마인드, 전담 상담)을 적용하고 싶다.
> **제약**: 그룹은 운영팀이 기억하고 관리할 수 있을 만큼 적어야 한다(5개 이내). 그룹이 바뀌면 정책도 바꿔야 하므로 자주 바뀌면 곤란하다.
> **질문**: 고객을 몇 개 그룹으로 나눠야 하고, 각 그룹은 어떤 고객이며, 그 그룹은 믿을 만한가?

## 이 실습의 질문

> **CRM팀에 줄 고객 그룹은 몇 개이고, 각 그룹의 기준은 무엇인가? 그 그룹이 "진짜"라고 CRM팀을 설득하는 근거는 무엇이고, 어디까지가 분석가의 선택인가?**

Chapter 1~5와 달리 이번에는 맞힐 정답이 없습니다. 그래서 "모델의 점수" 대신 **선택을 바꾸면 결과가 얼마나 바뀌는가**와 **군집에 쓰지 않은 데이터로 확인하기**에 시간을 씁니다.

## 이 노트북을 쓰는 법

| 표시 | 뜻 |
| --- | --- |
| **[직접 작성]** | 빈 코드 셀을 직접 채웁니다. 강의 교안의 코드를 보지 않고 먼저 써 보세요 |
| **[판단]** | 코드가 아니라 생각을 적는 칸입니다. 셀을 더블클릭해 `나의 답:` 뒤에 적으세요 |
| **[제공]** | 미리 채워 둔 코드입니다. 읽고 실행하세요 |
| **힌트** | 막혔을 때만 펼치세요 |
| **먼저 예상하기** | 결과를 보기 전에 예상을 적어 두고, 결과를 본 뒤 비교합니다 |

## 준비

UCI Online Retail II 데이터를 CSV로 바꾼 `online_retail_II.csv`(약 97MB, 1,067,371행)를 이 노트북과 같은 폴더에 두세요. 파일이 크므로 Colab에서는 Google Drive에 올려 두고 연결해서 읽는 것을 권합니다.

In [ ]:
# [제공] 이 실습에서 쓰는 라이브러리
import platform
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.metrics import silhouette_score, adjusted_rand_score

# 그래프의 한글 글꼴 (환경에 맞는 글꼴이 없으면 글자가 깨질 수 있습니다)
if platform.system() == "Windows":
    plt.rcParams["font.family"] = "Malgun Gothic"
elif platform.system() == "Darwin":
    plt.rcParams["font.family"] = "AppleGothic"
plt.rcParams["axes.unicode_minus"] = False

---
# Part 1. 고객을 숫자로: 정제와 RFM

## 1-1. 데이터 불러오기

**[제공]** 송장 번호와 상품 코드는 글자로, 주문 일시는 날짜로 읽습니다. 행이 100만 개가 넘어 읽는 데 수십 초가 걸릴 수 있습니다.

In [ ]:
# [제공] 데이터 불러오기
raw = pd.read_csv("online_retail_II.csv", dtype={"Invoice": str, "StockCode": str}, parse_dates=["InvoiceDate"])
print(raw.shape)
raw.head()

## 1-2. 정제: 실제로 산 기록만 남기기

**[직접 작성]** 다음 네 가지의 비율을 각각 출력한 뒤, 네 가지를 모두 뺀 데이터를 `sales`에 저장하세요. `sales`의 행 수와 고객 수(`CustomerID`의 고유값 수)도 출력합니다.

| 정제 항목 | 조건 |
| --- | --- |
| 고객 ID 없음 | `CustomerID`가 빈 값 |
| 취소 송장 | `Invoice`가 `C`로 시작 |
| 수량 0 이하 | `Quantity <= 0` |
| 가격 0 이하 | `Price <= 0` |

그다음 `sales`에 구매 금액 열 `amount`(= `Quantity` × `Price`)를 만드세요.

<details><summary>힌트</summary>

취소 송장은 `raw["Invoice"].str.startswith("C")`, 빈 값은 `raw["CustomerID"].isna()`. 비율은 조건에 `.mean()`을 붙입니다.
</details>

In [ ]:
print("고객 ID 없음 비율:", round(raw["CustomerID"].isna().mean(), 4))
print("취소 송장 비율:", round(raw["Invoice"].str.startswith("C").mean(), 4))
print("수량 0 이하 비율:", round((raw["Quantity"] <= 0).mean(), 4))
print("가격 0 이하 비율:", round((raw["Price"] <= 0).mean(), 4))

keep = raw["CustomerID"].notna() & ~raw["Invoice"].str.startswith("C") & (raw["Quantity"] > 0) & (raw["Price"] > 0)
sales = raw[keep].copy()
sales["amount"] = sales["Quantity"] * sales["Price"]
print("sales 행 수:", len(sales), "| 고객 수:", sales["CustomerID"].nunique())

> **[판단]** 반품을 빼지 않고 구매액에서 빼는 방식으로 정제했다면 군집 결과에 어떤 차이가 생길 수 있을까요? 이 정제 기준을 보고서에 적어야 하는 이유는 무엇인가요?
>
> 나의 답: 반품(취소 송장)을 구매액에서 빼는 방식이면 고객의 M이 줄거나 음수가 되어, 같은 고객이 다른 그룹으로 옮겨 갈 수 있습니다. 이 정제 기준은 군집 결과를 직접 바꾸므로, 다른 사람이 같은 결과를 재현하려면 보고서에 반드시 적어야 합니다.


## 1-3. 기간 나누기와 RFM 계산

| 구간 | 기간 | 용도 |
| --- | --- | --- |
| 군집용 | 2009-12-01 ~ 2010-11-30 | 고객을 묶는다 |
| 검증용 | 2010-12-01 ~ 2011-11-30 | 묶은 그룹이 다음 해에 실제로 다르게 행동했는지 확인한다 |

**[직접 작성]** `sales`를 두 기간으로 나눠 `period1`, `period2`에 저장하세요. 그다음 **군집용 기간(`period1`)으로만** 고객마다 RFM을 계산해 `rfm`(고객 ID가 인덱스, 열 `R`, `F`, `M`)에 저장하고, 고객 수를 출력하세요. 기준일은 2010-12-01입니다.

| 지표 | 계산 |
| --- | --- |
| `R` | 기준일 − 마지막 구매일 (일) |
| `F` | 송장 수 (`Invoice`의 고유값 수) |
| `M` | 구매 금액의 합 (`amount`의 합) |

<details><summary>힌트</summary>

```python
ref = pd.Timestamp("2010-12-01")
rfm = period1.groupby("CustomerID").agg(
    R=("InvoiceDate", lambda s: (ref - s.max()).days),
    F=("Invoice", "nunique"),
    M=("amount", "sum"),
)
```
</details>

In [ ]:
period1 = sales[(sales["InvoiceDate"] >= "2009-12-01") & (sales["InvoiceDate"] < "2010-12-01")].copy()
period2 = sales[(sales["InvoiceDate"] >= "2010-12-01") & (sales["InvoiceDate"] < "2011-12-01")].copy()
print("period1:", len(period1), "행 | period2:", len(period2), "행")

ref = pd.Timestamp("2010-12-01")
rfm = period1.groupby("CustomerID").agg(
    R=("InvoiceDate", lambda s: (ref - s.max()).days),
    F=("Invoice", "nunique"),
    M=("amount", "sum"),
)
print("군집 대상 고객 수:", len(rfm))

**[직접 작성]** `rfm`의 분포를 확인하세요. 최솟값, 중앙값, 평균, 상위 10%, 상위 1%, 최댓값, 표준편차가 보이도록 출력합니다.

<details><summary>힌트</summary>

`rfm.describe(percentiles=[0.5, 0.9, 0.99]).round(1)`
</details>

In [ ]:
rfm.describe(percentiles=[0.5, 0.9, 0.99]).round(1)

> **[판단]** F와 M의 평균과 중앙값은 얼마나 다른가요? 이 차이는 분포가 어떤 모양이라는 뜻인가요? 거리로 고객을 묶을 때 이 모양이 어떤 문제를 만들 것 같은가요?
>
> 나의 답: F와 M의 평균은 중앙값보다 훨씬 큽니다. 소수의 대량 구매 고객이 있는 오른쪽 꼬리 분포라는 뜻입니다. 거리로 묶으면 M의 큰 값이 거리를 지배해서, 금액이 큰 몇 명이 그룹 구조를 좌우하게 됩니다. 로그 변환과 표준화가 필요한 이유입니다.


---
# Part 2. K-means로 묶기

## 2-1. 손으로 돌려 보기

교안 2-1의 숫자 6개(1, 2, 3, 10, 11, 12)를 $k = 2$로 나눠 봅니다.

**[직접 작성]** `KMeans`에 초기 중심을 1과 2로 직접 주고(`init=np.array([[1], [2]])`, `n_init=1`) 학습한 뒤, 최종 중심, 그룹, inertia를 출력하세요. 이어서 같은 6개 점에 대해 `silhouette_score`로 실루엣 점수를 계산하세요.

In [ ]:
from sklearn.cluster import KMeans

x = np.array([[1], [2], [3], [10], [11], [12]])
km_hand = KMeans(n_clusters=2, init=np.array([[1], [2]]), n_init=1).fit(x)
print("최종 중심:", km_hand.cluster_centers_.ravel())
print("그룹:", km_hand.labels_)
print("inertia:", round(km_hand.inertia_, 2))
print("실루엣 점수:", round(silhouette_score(x, km_hand.labels_), 3))

> **[판단]** 나쁜 초기값(1과 2)에서 시작했는데도 최종 결과는 눈으로 봐도 자연스러운 두 덩어리인가요? 교안 2-1의 손 계산(중심 2와 11, inertia 4.0)과 같은가요?
>
> 나의 답: 네. 초기 중심이 1과 2로 나빴는데도 [1, 2, 3]과 [10, 11, 12] 두 덩어리로 수렴했습니다. 최종 중심은 2와 11, inertia는 4.0으로 교안의 손 계산과 같습니다.


## 2-2. 전처리가 군집을 정한다

### 먼저 예상하기

결과를 보기 전에 예상을 적어 두고, 결과를 본 뒤 비교합니다. 다음 절에서 같은 K-means($k = 4$)를 세 가지 전처리에 적용합니다.

> **[판단]** 예상: RFM 원본 그대로 K-means를 돌리면 네 그룹의 크기는 어떻게 나올까요? (예: 비슷한 크기 네 개 / 하나는 아주 크고 나머지는 작음)
>
> 나의 답: 예상: 원본 RFM에 K-means를 돌리면 한 그룹에 고객이 거의 몰리고 나머지는 작을 것이다. 결과를 보면 원본 전처리에서 한 그룹이 대부분을 차지해 예상이 맞았습니다.


**[제공]** K-means를 돌려 그룹 크기와 실루엣을 보여 주는 함수입니다. `n_init=10`(여러 번 시작해 가장 좋은 결과), `random_state=42`로 고정합니다.

In [ ]:
# [제공] K-means를 돌리고 그룹 크기와 실루엣을 출력
def run_kmeans(X, k, seed=42, verbose=True):
    km = KMeans(n_clusters=k, n_init=10, random_state=seed).fit(X)
    if verbose:
        print(f"k={k} | 그룹 크기 {np.bincount(km.labels_)} | inertia {km.inertia_:,.0f} | 실루엣 {silhouette_score(X, km.labels_):.3f}")
    return km

**[직접 작성]** 세 가지 전처리로 행렬을 만들고 각각 `run_kmeans(X, 4)`를 실행하세요.

1. `X_raw`: R, F, M 원본 그대로
2. `X_std`: 세 열을 `StandardScaler`로 표준화
3. `X_log`: F와 M에 `np.log1p`를 적용한 뒤(R은 그대로) 세 열을 표준화

<details><summary>힌트</summary>

```python
X_log = StandardScaler().fit_transform(np.column_stack([rfm["R"], np.log1p(rfm["F"]), np.log1p(rfm["M"])]))
```
</details>

In [ ]:
X_raw = rfm[["R", "F", "M"]].values
X_std = StandardScaler().fit_transform(rfm[["R", "F", "M"]])
X_log = StandardScaler().fit_transform(np.column_stack([rfm["R"], np.log1p(rfm["F"]), np.log1p(rfm["M"])]))

for name, X in [("원본", X_raw), ("표준화", X_std), ("로그+표준화", X_log)]:
    print(name, end=" | ")
    run_kmeans(X, 4)

> **[판단]** 예상은 맞았나요? 실루엣이 가장 높은 전처리는 무엇이고, 그 결과를 CRM팀에 줄 수 있나요? 실루엣이 높은데도 쓸 수 없다면 그 이유는 무엇인가요?
>
> 나의 답: 실습 데이터에서는 원본 전처리의 실루엣이 가장 높았지만, 한 그룹에 고객이 대부분 몰려 있어서 CRM팀에 줄 수 없습니다. 실루엣은 '잘 떨어져 있는가'만 보고 '그룹이 쓸모 있는가'는 보지 않습니다. 로그 후 표준화한 결과는 실루엣이 낮아도 그룹 크기가 고르게 나뉘므로 정책에 쓸 수 있습니다.


## 2-3. $k$ 고르기

**[직접 작성]** 로그 + 표준화한 `X_log`로 $k$를 2부터 8까지 바꿔 가며 `run_kmeans`를 실행하세요. inertia가 줄어드는 양(직전 $k$ 대비)도 계산해 보세요.

In [ ]:
rows, prev = [], None
for k in range(2, 9):
    km = run_kmeans(X_log, k, verbose=False)
    rows.append({
        "k": k,
        "inertia": round(km.inertia_),
        "직전 대비 감소": None if prev is None else round(prev - km.inertia_),
        "실루엣": round(silhouette_score(X_log, km.labels_), 3),
        "최소 그룹 크기": int(np.bincount(km.labels_).min()),
    })
    prev = km.inertia_
pd.DataFrame(rows)

> **[판단]** 엘보(inertia가 크게 줄다 완만해지는 지점)와 실루엣이 허용하는 후보는 무엇인가요? 숫자만으로 최종 $k$를 정할 수 있나요? 의뢰서의 제약(5개 이내, 그룹별로 다른 정책)을 넣으면 무엇을 고르겠습니까?
>
> 나의 답: 엘보는 k=2에서 4 사이에서 inertia가 크게 줄고, 그 이후에는 감소폭이 완만해집니다. 실루엣은 작은 k일수록 높아서 k=2가 최고입니다. 숫자만으로는 최종 k를 정할 수 없습니다. 의뢰서의 제약(5개 이내, 정책 관리)과 엘보를 함께 보면 3~4가 후보이고, 두 그룹은 정책 차별화에 너무 거칩니다. 그래서 k=4를 고르겠습니다.


## 2-4. 프로필과 이름 붙이기

**[직접 작성]** $k = 4$로 최종 모델을 만들어 그룹 번호를 `rfm["cluster"]`에 저장하고, 그룹별 고객 수와 R, F, M의 **중앙값**을 표로 출력하세요.

In [ ]:
km4 = run_kmeans(X_log, 4)
rfm["cluster"] = km4.labels_
profile = rfm.groupby("cluster").agg(
    고객수=("R", "size"),
    R중앙값=("R", "median"),
    F중앙값=("F", "median"),
    M중앙값=("M", "median"),
)
profile

**[직접 작성]** 위 프로필 표를 보고 그룹마다 이름을 붙여 `names` 딕셔너리에 저장하세요. 그룹 번호는 실행할 때마다 다를 수 있으니, **반드시 위 표를 보고** 번호와 이름을 맞춥니다. 이 이름은 Part 3에서 표를 읽기 쉽게 하는 데 씁니다.

예: `names = {0: "최근 1회 구매", 1: "핵심 고객", 2: "휴면 1회 구매", 3: "성장 중"}`

In [ ]:
names = {0: "핵심 고객", 1: "최근 1회 구매", 2: "휴면 1회 구매", 3: "성장 중"}

**[제공]** 이름을 붙인 그룹 열 `group`을 만듭니다. (위 셀에서 이미 만들었다면 다시 실행해도 같습니다.)

In [ ]:
# [제공] 이름 붙인 그룹 열
rfm["group"] = rfm["cluster"].map(names)
rfm["group"].value_counts()

> **[판단]** 이름은 어떤 숫자를 근거로 붙였나요? 같은 표를 보고 다른 분석가는 다른 이름을 붙일 수 있을까요? 이름을 보고서에 쓸 때 함께 적어야 할 것은 무엇인가요?
>
> 나의 답: 이름은 그룹의 중앙값 R(최근성), F(구매 횟수), M(구매 금액)을 보고 붙였습니다. 다른 분석가도 '최근'을 며칠로 볼지에 따라 다른 이름을 붙일 수 있습니다. 보고서에는 이름과 함께 그 이름의 근거 숫자, k, 전처리 방식, random_state를 적어야 합니다. 그룹 번호는 실행할 때마다 바뀔 수 있기 때문입니다.


---
# Part 3. 묶고 의심하기

## 3-1. 선택을 바꾸면 그룹이 바뀌는가

### 먼저 예상하기

결과를 보기 전에 예상을 적어 두고, 결과를 본 뒤 비교합니다.

> **[판단]** 예상: $k = 3$의 그룹들은 $k = 4$에서 "하나가 둘로 쪼개지는" 모양일까요, 아니면 더 복잡하게 섞일까요? 그리고 초기값만 바꿔 다시 돌리면 결과가 달라질까요?
>
> 나의 답: 예상은 빗나갔습니다. k=3의 한 그룹은 k=4에서 두 그룹으로 나뉘었고, 다른 그룹은 휴면과 최근 1회 구매가 섞이는 등 단순히 쪼개지기만 한 게 아니라 재배열되었습니다. 초기값(random_state 0~4)을 바꾸면 ARI가 0.98~0.999로 거의 같은 결과가 나옵니다. 그러나 계층적 군집과의 ARI는 0.39로 낮습니다.


**[직접 작성]** 세 가지를 확인하세요.

1. $k = 3$으로 돌린 결과를 `rfm["k3"]`에 저장하고, `pd.crosstab(rfm["k3"], rfm["group"])`으로 $k = 4$의 그룹과 겹쳐 보기 ($k = 3$ 그룹의 R, F, M 중앙값도 함께 출력)
2. $k = 4$를 `random_state` 0~4로 다섯 번 다시 돌려, 최종 모델(`km4`)과의 ARI(`adjusted_rand_score`)
3. 계층적 군집(`AgglomerativeClustering(n_clusters=4, linkage="ward")`)과 최종 모델의 ARI

<details><summary>힌트</summary>

`run_kmeans(X_log, 4, seed=s, verbose=False).labels_`로 다시 돌린 결과를 얻고, `adjusted_rand_score(km4.labels_, 그 결과)`로 비교합니다.
</details>

In [ ]:
k3 = run_kmeans(X_log, 3)
rfm["k3"] = k3.labels_
print(pd.crosstab(rfm["k3"], rfm["group"]))
print(rfm.groupby("k3")[["R", "F", "M"]].median().round(1))

aris = []
for s in range(5):
    labels = run_kmeans(X_log, 4, seed=s, verbose=False).labels_
    aris.append(adjusted_rand_score(km4.labels_, labels))
print("random_state 0~4 와의 ARI:", [round(a, 3) for a in aris])

agg = AgglomerativeClustering(n_clusters=4, linkage="ward").fit(X_log)
print("계층적 군집과의 ARI:", round(adjusted_rand_score(km4.labels_, agg.labels_), 3))

> **[판단]** 예상은 맞았나요? $k = 3$의 한 그룹이 $k = 4$에서 어떻게 나뉘고 합쳐지는지 숫자로 설명하세요. 초기값에 대한 ARI와 계층적 군집과의 ARI를 보고, 이 군집이 "안정적"인지와 "유일한 정답"인지를 구분해 말해 보세요.
>
> 나의 답: k=3의 그룹 중 하나는 k=4에서 '성장 중'과 '최근 1회 구매'로 나뉘고, 다른 그룹은 '휴면 1회 구매'와 일부 고객이 섞입니다. 초기값에 대한 ARI가 높으니 이 k=4 결과는 같은 방법을 다시 돌리면 안정적입니다. 하지만 계층적 군집과 ARI가 낮으니 '유일한 정답'이라고 말할 수는 없습니다. 안정적인 것과 유일한 것은 다릅니다.


## 3-2. 다음 해에 정말 달랐는가

정답이 없으니 "군집이 맞는가"는 물을 수 없습니다. 대신 "**군집이 앞으로의 행동을 가르는가**"를 봅니다. 검증용 기간(`period2`)은 군집에 한 번도 쓰지 않았습니다.

**[직접 작성]** 고객마다 검증용 기간의 구매 금액 합을 `rfm["next_amount"]`(구매가 없으면 0), 재구매 여부를 `rfm["repurchase"]`(구매 금액 > 0)에 저장하고, 그룹별로 고객 수, 다음 해 재구매율, 구매액 평균, 구매액 중앙값을 출력하세요. 전체 재구매율도 출력합니다.

<details><summary>힌트</summary>

`period2.groupby("CustomerID")["amount"].sum()`을 `.reindex(rfm.index).fillna(0)`으로 군집 대상 고객에 맞춥니다.
</details>

In [ ]:
rfm["next_amount"] = period2.groupby("CustomerID")["amount"].sum().reindex(rfm.index).fillna(0)
rfm["repurchase"] = rfm["next_amount"] > 0
summary = rfm.groupby("group").agg(
    고객수=("R", "size"),
    재구매율=("repurchase", "mean"),
    구매액평균=("next_amount", "mean"),
    구매액중앙값=("next_amount", "median"),
)
print(summary.round(3))
print("전체 재구매율:", round(rfm["repurchase"].mean(), 3))

> **[판단]** 다음 해 재구매율은 그룹마다 얼마나 다른가요? 특히 "휴면 1회 구매"와 "최근 1회 구매"는 F와 M이 거의 같은데 재구매율은 어떤가요? 이 결과는 $k = 4$를 고른 선택에 대해 무엇을 말해 주나요?
>
> 나의 답: 재구매율은 그룹마다 분명히 다릅니다. 핵심 고객이 가장 높고, 휴면 1회 구매가 가장 낮습니다. '최근 1회 구매'와 '휴면 1회 구매'는 F와 M이 비슷하지만 재구매율은 다릅니다. 두 그룹을 가르는 것은 구매 금액이 아니라 최근성(R)이라는 뜻입니다. 이 결과는 k=4를 고른 근거가 되지만, 그 k가 최적이라는 증명은 아닙니다.


> **[판단]** 이 검증 결과가 말하지 **않는** 것을 두 가지 적어 보세요. (예: 그룹이 유일한가? 그룹이 원인인가? 정책이 효과가 있는가?)
>
> 나의 답: 이 검증이 말하지 않는 것: (1) 이 그룹 구분이 유일하다는 것, (2) 그룹이 재구매를 일으킨다는 인과, (3) 그룹별로 다른 정책(할인, 리마인드)이 실제로 효과가 있다는 것.


## 3-3. 사람의 규칙과 비교하기

CRM 현장에서는 모델 없이 **RFM 점수**로 고객을 나눠 왔습니다. R, F, M을 고객 수가 같은 다섯 구간으로 나눠 1~5점을 주고(R은 최근일수록 5점), 점수 조합으로 세그먼트를 정합니다.

| 규칙 세그먼트 | 조건 |
| --- | --- |
| VIP | R 4~5점이고 F 4~5점 |
| 최근 신규 | R 4~5점 (F는 낮음) |
| 이탈 위험 우량 | R 1~2점이고 F 4~5점 |
| 휴면 | R 1~2점 (F는 낮음) |
| 일반 | 나머지 |

**[제공]** RFM 점수와 규칙 세그먼트를 만듭니다. F는 같은 값이 많아(절반이 2회 이하) 순위로 바꾼 뒤 다섯 구간으로 나눕니다.

In [ ]:
# [제공] RFM 점수와 규칙 세그먼트
r_score = pd.qcut(rfm["R"].rank(method="first"), 5, labels=[5, 4, 3, 2, 1]).astype(int)
f_score = pd.qcut(rfm["F"].rank(method="first"), 5, labels=[1, 2, 3, 4, 5]).astype(int)
conditions = [
    (r_score >= 4) & (f_score >= 4),
    (r_score >= 4),
    (r_score <= 2) & (f_score >= 4),
    (r_score <= 2),
]
rfm["rule"] = np.select(conditions, ["VIP", "최근 신규", "이탈 위험 우량", "휴면"], default="일반")
rfm["rule"].value_counts()

**[직접 작성]** 두 가지를 출력하세요.

1. 규칙 세그먼트별 고객 수, R·F·M 중앙값, 다음 해 재구매율, 다음 해 구매액 평균
2. `pd.crosstab(rfm["rule"], rfm["group"])`으로 규칙 세그먼트와 K-means 그룹 겹쳐 보기

In [ ]:
print(rfm.groupby("rule").agg(
    고객수=("R", "size"),
    R중앙값=("R", "median"),
    F중앙값=("F", "median"),
    M중앙값=("M", "median"),
    재구매율=("repurchase", "mean"),
    구매액평균=("next_amount", "mean"),
).round(2))
print(pd.crosstab(rfm["rule"], rfm["group"]))

> **[판단]** 규칙 세그먼트도 다음 해 행동을 가르나요? K-means의 "핵심 고객"과 규칙의 "VIP"는 어떻게 다른가요? K-means에는 없는데 규칙에는 있는 세그먼트는 무엇이고, 그 세그먼트는 CRM팀에 왜 중요할까요?
>
> 나의 답: 규칙 세그먼트도 다음 해 재구매율을 구분합니다. K-means의 '핵심 고객'은 거리로 묶은 그룹이고, 규칙의 VIP는 점수 컷으로 정한 그룹이라 경계가 다릅니다. K-means에는 없고 규칙에만 있는 '이탈 위험 우량'(구매 횟수와 금액은 높지만 최근에 안 산 고객)은 CRM팀에 중요합니다. 과거 우량 고객을 되찾을 대상이기 때문입니다.


> **[판단]** 의뢰서의 제약 "그룹이 자주 바뀌면 곤란하다"를 생각하면 K-means와 규칙 중 어느 쪽이 유리한가요? 둘을 함께 쓴다면 어떻게 쓰겠습니까?
>
> 나의 답: 의뢰서 제약을 생각하면 규칙이 유리합니다. 점수 컷은 데이터가 조금 바뀌어도 그룹이 거의 안 바뀌기 때문입니다. K-means는 데이터가 바뀌면 그룹 경계가 옮겨 갈 수 있습니다. 함께 쓴다면 규칙을 운영 기본으로 두고, K-means는 그룹 특성을 설명하고 분기마다 재검토할 때 보조로 쓰겠습니다.


---
# Part 4. CRM팀에 보고하기

## 4-1. 쓸 수 있는 문장과 없는 문장

다음 문장들이 이 실습의 결과로 **쓸 수 있는 문장인지** 판단하고, 쓸 수 없다면 이유를 적으세요.

1. "고객을 R·F·M 기준으로 4개 그룹으로 나눴다. 각 그룹의 기준값은 표와 같다."
2. "데이터 분석 결과 우리 고객은 본래 4개 유형으로 나뉜다."
3. "네 그룹은 다음 해 재구매율이 35%에서 95%까지 다르다."
4. "초기값을 바꿔도 결과가 같았으므로(ARI 1.0에 가까움) 이 군집은 유일한 정답이다."
5. "이탈 위험 고객은 K-means 결과에 나타나지 않으므로 우리 고객 중에는 없다."

> **[판단]** 1~5번 각각: 쓸 수 있다 / 쓸 수 없다(이유)
>
> 나의 답:
>
> 1. 쓸 수 있다. 기준값 표를 함께 보여 주면 됩니다.
> 2. 쓸 수 없다. 이 그룹은 이 데이터와 설정에서 나온 것이고, 고객이 본래 가진 유형이 아닙니다.
> 3. 실제 데이터의 표에서 범위를 확인한 뒤에 쓸 수 있습니다. 숫자를 임의로 적으면 안 됩니다.
> 4. 쓸 수 없다. 초기값에 안정적이어도 계층적 군집과 ARI가 낮을 수 있어서 유일한 정답이 아닙니다.
> 5. 쓸 수 없다. K-means 결과에 안 나타나도 규칙 세그먼트처럼 다른 기준으로는 존재할 수 있습니다.


## 4-2. CRM팀 보고 메모

분석 결과를 CRM팀장에게 메시지로 보낸다고 생각하고 쓰세요. 팀장은 머신러닝 용어를 모릅니다. 아래 네 가지가 모두 들어가야 합니다.

1. **그룹 수와 각 그룹의 기준**: 이름과 함께 R·F·M 기준 숫자
2. **분석가의 선택**: 어떤 정보로, 어떻게 가공해, 왜 4개로 나눴는지 (다른 선택이 가능했다는 것도)
3. **믿을 만하다는 근거**: 군집에 쓰지 않은 다음 해 행동
4. **제안과 주의할 점**: 그룹별 정책을 어떻게 시험할지, 그룹 구분을 얼마나 자주 다시 볼지, 규칙과 함께 쓰는 방법

> **[판단]** CRM팀 보고 메모:
>
> 1. 그룹 수와 기준:
>
> 2. 분석가의 선택:
>
> 3. 믿을 만하다는 근거:
>
> 4. 제안과 주의할 점:
>
> 나의 답:
>
> 1. 그룹 수와 기준: 4개(최근 1회 구매, 핵심 고객, 휴면 1회 구매, 성장 중). 각 그룹의 최근성(R, 일), 구매 횟수(F), 구매 금액(M) 중앙값은 [실제 데이터 표의 숫자를 적을 것].
> 2. 분석가의 선택: 2009-12~2010-11 구매로 군집을 만들었고, 취소·반품·고객 ID 없는 기록은 뺐습니다. 구매 횟수와 금액은 로그 변환 후 표준화했고, 그룹 수는 엘보와 '5개 이내' 제약을 함께 보고 4개로 골랐습니다. 다른 선택(3개, 다른 기간)도 가능했습니다.
> 3. 믿을 만하다는 근거: 군집에 쓰지 않은 다음 1년 구매에서, 그룹별 재구매율이 [실제 데이터 숫자]로 달랐습니다.
> 4. 제안과 주의할 점: 그룹별 정책은 고객 일부에게 무작위로 먼저 시험합니다. 그룹 구분은 분기마다 다시 보고, 번호가 아니라 기준으로 소통합니다. 규칙 세그먼트와 함께 봅니다.


---
# 마무리

## 이 Chapter의 판단 포인트

> **군집은 발견이 아니라 선택이다. 전처리와 $k$를 바꾸면 그룹이 바뀐다. 그룹이 쓸모 있는지는 다음 기간의 행동으로 확인한다.**

## 개인 검증 체크리스트에 추가

오늘 실습에서 내가 실제로 겪은 일을 바탕으로, 체크리스트의 줄을 내 말로 적으세요.

예: "군집 결과를 보면 '전처리는 무엇이었고 $k$는 왜 그 값인가'를 먼저 묻는다."

> **[판단]** 나의 체크리스트 10번(군집 결과를 받았을 때):
>
> 나의 답: 군집 결과를 받으면 전처리 방식, k를 고른 이유, 초기값에 따른 안정성을 먼저 확인합니다.


> **[판단]** 나의 체크리스트 11번(정답이 없는 결과를 검증할 때):
>
> 나의 답: 정답이 없는 결과는 다음 기간의 행동으로 검증하고, 다른 방법(규칙, 다른 알고리즘)과 비교해서 결과가 얼마나 달라지는지 봅니다.


## 제출물

- 이 노트북 (모든 **[판단]** 칸을 채운 상태)

수고했습니다. 다음 Chapter 7에서는 차원 축소, 이상 탐지, 장바구니 분석으로 정답 없는 데이터에서 패턴을 찾고, 그 패턴이 내가 고른 기준의 결과인지 확인합니다.